## Dataset

In [2]:
!pip install ucimlrepo

In [3]:
from ucimlrepo import fetch_ucirepo

# fetch dataset
online_shoppers_purchasing_intention_dataset = fetch_ucirepo(id=468)

# data (as pandas dataframes)
X = online_shoppers_purchasing_intention_dataset.data.features
y = online_shoppers_purchasing_intention_dataset.data.targets

# metadata
print(online_shoppers_purchasing_intention_dataset.metadata)

# variable information
print(online_shoppers_purchasing_intention_dataset.variables)


{'uci_id': 468, 'name': 'Online Shoppers Purchasing Intention Dataset', 'repository_url': 'https://archive.ics.uci.edu/dataset/468/online+shoppers+purchasing+intention+dataset', 'data_url': 'https://archive.ics.uci.edu/static/public/468/data.csv', 'abstract': 'Of the 12,330 sessions in the dataset,\n84.5% (10,422) were negative class samples that did not\nend with shopping, and the rest (1908) were positive class\nsamples ending with shopping.', 'area': 'Business', 'tasks': ['Classification', 'Clustering'], 'characteristics': ['Multivariate'], 'num_instances': 12330, 'num_features': 17, 'feature_types': ['Integer', 'Real'], 'demographics': [], 'target_col': ['Revenue'], 'index_col': None, 'has_missing_values': 'no', 'missing_values_symbol': None, 'year_of_dataset_creation': 2018, 'last_updated': 'Thu Jan 11 2024', 'dataset_doi': '10.24432/C5F88Q', 'creators': ['C. Sakar', 'Yomi Kastro'], 'intro_paper': {'ID': 367, 'type': 'NATIVE', 'title': 'Real-time prediction of online shoppers’ pur

### Step 1: Data Preprocessing and Splitting
We will preprocess the features (encode categorical columns, scale numerical features) and split the data into training and test sets.

In [4]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Combine X and y momentarily to understand features or handle targets
# Target variable 'Revenue' (y) is boolean, let's map it to integers (0 and 1)
y_numeric = y['Revenue'].astype(int)

# Identify numerical and categorical columns
num_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
cat_cols = X.select_dtypes(include=['object', 'bool']).columns.tolist()

# Define preprocessing pipelines
num_transformer = Pipeline(steps=[
    ('scaler', StandardScaler())
])

cat_transformer = Pipeline(steps=[
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', num_transformer, num_cols),
        ('cat', cat_transformer, cat_cols)
    ]
)

# Split the dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y_numeric, test_size=0.2, random_state=42, stratify=y_numeric)

# Fit and transform training data, transform testing data
X_train_preprocessed = preprocessor.fit_transform(X_train)
X_test_preprocessed = preprocessor.transform(X_test)

# Get feature names after one-hot encoding for reference if needed
cat_encoder = preprocessor.named_transformers_['cat'].named_steps['onehot']
encoded_cat_cols = cat_encoder.get_feature_names_out(cat_cols).tolist()
all_features = num_cols + encoded_cat_cols

print(f"Preprocessed training shape: {X_train_preprocessed.shape}")
print(f"Preprocessed testing shape: {X_test_preprocessed.shape}")

Preprocessed training shape: (9864, 29)
Preprocessed testing shape: (2466, 29)


### Step 2: Model Training and Evaluation
We will train the 5 specified classifiers and compute the required evaluation metrics: Accuracy, AUC Score, Precision, Recall, F1 Score, and MCC (Matthews Correlation Coefficient).

In [5]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, roc_auc_score, precision_score, recall_score, f1_score, matthews_corrcoef
import joblib
import os

# Create directory to save models
os.makedirs('model', exist_ok=True)

# Initialize models
models = {
    'Logistic_Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Decision_Tree': DecisionTreeClassifier(max_depth=6, random_state=42),
    'KNN': KNeighborsClassifier(n_neighbors=5),
    'Naive_Bayes': GaussianNB(),
    'SVM': SVC(probability=True, random_state=42)  # probability=True required for AUC calculation
}

results = {}

# Train and evaluate
for name, model in models.items():
    # Train
    model.fit(X_train_preprocessed, y_train)

    # Predict
    y_pred = model.predict(X_test_preprocessed)
    y_prob = model.predict_proba(X_test_preprocessed)[:, 1]

    # Metrics
    acc = accuracy_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_prob)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    mcc = matthews_corrcoef(y_test, y_pred)

    results[name] = {
        'Accuracy': acc,
        'AUC': auc,
        'Precision': prec,
        'Recall': rec,
        'F1_Score': f1,
        'MCC': mcc
    }

    # Save models to disk
    joblib.dump(model, f'model/{name}.pkl')

# Display results table
results_df = pd.DataFrame(results).T
display(results_df)

,Accuracy,AUC,Precision,Recall,F1_Score,MCC
Logistic_Regression,0.881184,0.887636,0.743169,0.356021,0.481416,0.460316
Decision_Tree,0.893755,0.905990,0.697368,0.554974,0.618076,0.562200
KNN,0.873074,0.781761,0.645570,0.400524,0.494346,0.442196
Naive_Bayes,0.673560,0.793161,0.293659,0.787958,0.427861,0.323431
SVM,0.886861,0.873794,0.706827,0.460733,0.557845,0.511218


### Step 3: Export Artifacts
We will save the preprocessor pipeline and a chunk of the raw testing data as `test_data.csv` to satisfy the Streamlit repository requirements.

In [6]:
# Save the preprocessor pipeline
joblib.dump(preprocessor, 'model/preprocessor.pkl')

# Create a test CSV containing both features and targets for easily test-loading in Streamlit
test_df = X_test.copy()
test_df['Revenue'] = y_test
test_df.to_csv('test_data.csv', index=False)

print("Saved preprocessor to 'model/preprocessor.pkl'")
print("Saved test dataset preview to 'test_data.csv'")

Saved preprocessor to 'model/preprocessor.pkl'
Saved test dataset preview to 'test_data.csv'


In [11]:
!zip -r model /content/model

  adding: content/model/ (stored 0%)
  adding: content/model/Decision_Tree.pkl (deflated 60%)
  adding: content/model/preprocessor.pkl (deflated 55%)
  adding: content/model/KNN.pkl (deflated 84%)
  adding: content/model/Naive_Bayes.pkl (deflated 16%)
  adding: content/model/SVM.pkl (deflated 79%)
  adding: content/model/Logistic_Regression.pkl (deflated 26%)
